# Day 59 - temperature, top-k and top-p

A model's last act on every step is a row of logits, one per token. Turning that row
into a single token is the sampling step, and it is where quality is traded for variety.

The language model here is a character 8-gram trained on sentences from a grammar, so
"is this sample grammatical" has a real answer and every claim below is a measured rate.

In [ ]:
import numpy as np
from sampling import (NGram, make_corpus, softmax, entropy, temperature, top_k, top_p,
                      min_p, nucleus_size, generate, score, greedy, is_grammatical,
                      total_sentences, temperature_table, rescue_table, shape_table,
                      order_matters, pivot_search, sort_cost, forgot_to_renormalise,
                      typical, typical_vs_nucleus, typical_contrast, all_sentences,
                      sentence_logprob, sequence_table)

text = make_corpus()
m = NGram(text)
print(len(text), 'characters,', text.count(chr(10)), 'sentences')
print('vocabulary', m.V, 'characters')
print(total_sentences(), 'legal sentences in the grammar')

47433 characters, 1500 sentences
vocabulary 24 characters
1250 legal sentences in the grammar


## 1. One step, and the tail

The top three characters hold almost everything. Almost.

In [ ]:
ctx = 'the small cat '
p = softmax(m.logits(ctx))
order = np.argsort(-p)
for i in order[:5]:
    print(repr(m.vocab[i]), round(float(p[i]), 4))
nuc = nucleus_size(p, 0.9)
tail = float(p[order[nuc:]].sum())
print('tail mass      ', round(tail, 6))
print('entropy        ', round(entropy(p), 3), 'bits')
print('nucleus at 0.9 ', nuc)
print('chance of hitting the tail over a 30-step sentence:',
      round(1 - (1 - tail) ** 30, 4))

's' 0.6099
'r' 0.2232
'w' 0.1637
'\n' 0.0001
'b' 0.0001
tail mass       0.003123
entropy         1.385 bits
nucleus at 0.9  3
chance of hitting the tail over a 30-step sentence: 0.0896


## 2. Greedy: correct, and only one sentence

In [ ]:
print(repr(greedy(m)))
rng = np.random.default_rng(0)
lines, _ = generate(m, rng, n=60, t=0.0)
print(score(lines))

'\nthe red cat sang in the garden.\n'
{'n': 60, 'grammatical': 1.0, 'distinct': 1}


## 3. Greedy is not even the most likely sentence

The grammar admits 1,250 sentences, so every one of them can be scored exactly and
greedy's output can be ranked among them. Step-wise argmax is not sequence argmax.

In [ ]:
sq = sequence_table(m)
print('best    ', repr(sq['best']), round(sq['best_lp'], 3))
print('greedy  ', repr(sq['greedy']), round(sq['greedy_lp'], 3))
print('rank    ', sq['rank'], 'of', sq['n'])
print('legal sentences hold', round(sq['legal_mass'], 4), 'of the sentence-level mass')
print('expected distinct in 100 draws:', round(sq['expected_distinct'], 1))

best     'a small dog sat in the garden.' -6.244
greedy   'the red cat sang in the garden.' -6.717
rank     44 of 1250
legal sentences hold 0.9088 of the sentence-level mass
expected distinct in 100 draws: 87.3


## 4. Temperature

`logits / t`, then softmax. `t = 0` is a separate argmax path, never a division.

In [ ]:
print(' t    entropy  grammatical  distinct')
for r in temperature_table(m):
    print(f"{r['t']:4.1f}  {r['H']:7.3f}  {r['grammatical']:10.0%}  {r['distinct']:8d}")

 t    entropy  grammatical  distinct
 0.0    0.000        100%         1
 0.3    0.200        100%        79
 0.7    0.269        100%        98
 1.0    0.336         90%        88
 1.3    0.557         55%        53
 1.7    1.182          5%         5
 2.0    1.787          1%         1


## 4. Truncation at a fixed temperature

Every row below is `t = 2.0` - the setting that scored worst above - plus one rule.

In [ ]:
for r in rescue_table(m):
    print(f"{r['name']:18} {r['grammatical']:6.0%}  {r['distinct']:4d} distinct")

no truncation          0%     0 distinct
top-k 1 (greedy)     100%     2 distinct
top-k 3                0%     0 distinct
top-p 0.9              2%     2 distinct
top-p 0.7             89%    85 distinct
top-p 0.5            100%    80 distinct
min-p 0.1            100%    96 distinct
typical 0.9            2%     2 distinct
typical 0.5           94%    84 distinct


## 5. Why a mass beats a count

A fixed `k` cannot be right on both a peaked step and a flat one.

In [ ]:
st = shape_table(m)
print('peaked steps', st['peaked']['n'], ' flat steps', st['flat']['n'])
print('mean nucleus (p=0.9):', round(st['peaked']['nucleus'], 2),
      'vs', round(st['flat']['nucleus'], 2))
for k in st['ks']:
    print(f"k={k:<4} mass left out: peaked {st['peaked']['tail_in_k'][k]:.4f}"
          f"   flat {st['flat']['tail_in_k'][k]:.4f}")

peaked steps 249  flat steps 61
mean nucleus (p=0.9): 1.0 vs 3.46
k=1    mass left out: peaked 0.0045   flat 0.5413
k=2    mass left out: peaked 0.0041   flat 0.2598
k=5    mass left out: peaked 0.0035   flat 0.0140
k=20   mass left out: peaked 0.0007   flat 0.0005


## 6. Typical sampling

A third question: keep the tokens whose surprise is closest to this step's entropy.
On a 24-character vocabulary it never separates from top-p, so the contrast is shown
on a distribution built for it.

In [ ]:
print(typical_vs_nucleus(m))
tc = typical_contrast()
print('entropy', round(tc['H'], 2), 'nats')
for r in tc['rows']:
    print(f"  {r['name']:20} surprise {r['surprise']:5.2f}  distance {r['distance']:5.2f}")
print('top-p keeps', tc['top_p_n'], 'and keeps the best:', tc['top_p_keeps_best'])
print('typical keeps', tc['typical_n'], 'and keeps the best:', tc['typical_keeps_best'])

{'steps': 1875, 'same_size': 1875, 'dropped_argmax': np.int64(0), 'mass': 0.9}
entropy 2.28 nats
  dominant, p=0.60     surprise  0.51  distance  1.77
  middle, p=0.03       surprise  3.51  distance  1.23
  tail, p=0.0001       surprise  9.21  distance  6.93
top-p keeps 1 and keeps the best: True
typical keeps 9 and keeps the best: False


## 7. The order of operations

`temperature -> top-p` and `top-p -> temperature` are different distributions.

In [ ]:
o = order_matters(m)
print(o)

{'ctx': 'the small cat s', 't': 2.5, 'p': 0.9, 'n_a': 18, 'n_b': 2, 'max_a': 0.3942865215556097, 'max_b': 0.5444882590872145, 'tv': 0.2758585424475533}


## 8. Sampling on a GPU without sorting

top-p can be rewritten as "find the probability threshold whose surviving mass reaches
p". That is monotone in one variable, so it bisects - a reduction per round, no sort.

In [ ]:
q = softmax(m.logits('\nthe '))
print(pivot_search(q), 'sorted answer:', nucleus_size(q, 0.9))
print(sort_cost())
print('forgetting to renormalise:', forgot_to_renormalise(m))

{'n': 5, 'mass': 0.9117744567989391, 'pivot': 0.07141945403558336} sorted answer: 5
{'V': 151936, 'k': 50, 'full': 2615290.202715513, 'partial': 152218.19280948874, 'ratio': 17.18119335438914}
forgetting to renormalise: {'kept_mass': 0.9968768590124926, 'overflow': 0.00265, 'last_token': 's'}


## 9. Tests

In [ ]:
p = softmax(m.logits('the small cat '))
assert abs(top_k(p, 3).sum() - 1) < 1e-12 and int((top_k(p, 3) > 0).sum()) == 3
assert abs(top_p(p, 0.9).sum() - 1) < 1e-12
assert int((top_p(p, 0.0) > 0).sum()) == 1
assert entropy(softmax(temperature(m.logits('the '), 0.5))) < \
       entropy(softmax(temperature(m.logits('the '), 2.0)))
assert is_grammatical('the small cat sat on the mat.')
assert abs(typical(p, 0.9).sum() - 1) < 1e-12
tc = typical_contrast()
assert tc['top_p_keeps_best'] and not tc['typical_keeps_best']
assert not is_grammatical('the small cat sat on the moon.')
print('all assertions passed')

all assertions passed
